# Stonekey Performance Diagnostics

**Author:** @haewon.yum  
**Client:** Netmarble — Stonekey  
**Purpose:** Campaign efficiency diagnostics + monetization timing analysis for D1/D3 model eligibility

---

## Sections
0. Verify KPI Event
1. Campaign-Level Performance (past 7 days): CPI, Login CPA, D1/D7 ROAS by OS
2. Purchasing Behavior — Revenue Timeline (D1→D30 cumulative curve)
3. D1/D3 Model Recommendation

## Key Tables
- `moloco-ae-view.athena.fact_dsp_core` — campaign performance snapshots
- `focal-elf-631.prod_stream_view.cv` — event-level conversion data

In [95]:
#@title Environment Setup

from google.cloud import bigquery
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from IPython.display import display
import warnings

warnings.filterwarnings('ignore')

pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 200)
pd.set_option('display.max_colwidth', 80)

client = bigquery.Client(project='moloco-ods')

def run_query(query, label=''):
    """Run a BQ query and return DataFrame. Print row count."""
    try:
        df = client.query(query).result().to_dataframe()
        status = f'✅ {label}: {len(df)} rows' if len(df) > 0 else f'⚠️ {label}: 0 rows — check needed'
        print(status)
        return df
    except Exception as e:
        print(f'❌ {label}: Query failed — {e}')
        return pd.DataFrame()

def sql_in_clause(values, field):
    """Generate SQL IN clause fragment for a list of string values."""
    if not values:
        return ''
    quoted = ', '.join(f"'{v}'" for v in values)
    return f'AND {field} IN ({quoted})'

In [96]:
#@title Parameters — Edit before running

BUNDLE_IDS_FACT = ['com.netmarble.stonkey', '6737408689']   # for fact_dsp_core (iOS stored without 'id' prefix)
BUNDLE_IDS_CV   = ['com.netmarble.stonkey', 'id6737408689'] # for cv.pb.app.bundle (iOS stored with 'id' prefix)

# Analysis windows
ANALYSIS_WINDOW_DAYS = 7       # Section 1: campaign performance lookback
COHORT_WINDOW_DAYS   = 14      # Section 2: install cohort lookback (days since launch)
COHORT_MAX_DAY       = 7       # Section 2: max days-since-install to show

# Section 2: countries to break down (in addition to global)
COHORT_COUNTRIES = ['KOR', 'USA', 'JPN']

# KPI events for Login CPA — both event names used across Android/iOS
KPI_EVENTS = ['login_1st', 'login_first']

# Eligibility thresholds
D1_THRESHOLD = 0.50
D3_THRESHOLD = 0.50

# Eligibility tracker spreadsheet
ELIGIBILITY_TRACKER_URL = 'https://docs.google.com/spreadsheets/d/1w8StJ19HpuPZ8kj4oA3SD_To5sEoEVpoZce8gpkthQY/edit?gid=1375953392'

# Derived helpers
_kpi_in          = ', '.join(f"'{e}'" for e in KPI_EVENTS)
_bundle_fact_in  = ', '.join(f"'{b}'" for b in BUNDLE_IDS_FACT)
_bundle_cv_in    = ', '.join(f"'{b}'" for b in BUNDLE_IDS_CV)
_cohort_ctry_in  = ', '.join(f"'{c}'" for c in COHORT_COUNTRIES)

print('Parameters set:')
print(f'  BUNDLE_IDS_FACT       = {BUNDLE_IDS_FACT}')
print(f'  BUNDLE_IDS_CV         = {BUNDLE_IDS_CV}')
print(f'  ANALYSIS_WINDOW_DAYS  = {ANALYSIS_WINDOW_DAYS}')
print(f'  COHORT_WINDOW_DAYS    = {COHORT_WINDOW_DAYS}')
print(f'  COHORT_MAX_DAY        = {COHORT_MAX_DAY}')
print(f'  COHORT_COUNTRIES      = {COHORT_COUNTRIES}')
print(f'  KPI_EVENTS            = {KPI_EVENTS}')

Parameters set:
  BUNDLE_IDS_FACT       = ['com.netmarble.stonkey', '6737408689']
  BUNDLE_IDS_CV         = ['com.netmarble.stonkey', 'id6737408689']
  ANALYSIS_WINDOW_DAYS  = 7
  COHORT_WINDOW_DAYS    = 14
  COHORT_MAX_DAY        = 7
  COHORT_COUNTRIES      = ['KOR', 'USA', 'JPN']
  KPI_EVENTS            = ['login_1st', 'login_first']


---
## Section 0: Verify KPI Event

In [97]:
#@title 0. Verify KPI event names from fact_dsp_all

q_kpi = f"""
SELECT
  campaign.os                     AS os,
  LOWER(event.name)               AS event_name,
  SUM(conversions)                AS conversions
FROM `moloco-ae-view.athena.fact_dsp_all`
WHERE product.app_market_bundle IN ({_bundle_fact_in})
  AND DATE(timestamp_utc) >= DATE_SUB(CURRENT_DATE(), INTERVAL 30 DAY)
  AND event.name IS NOT NULL
GROUP BY 1, 2
ORDER BY conversions DESC
LIMIT 30
"""

df_kpi = run_query(q_kpi, 'Post-install event names (fact_dsp_all)')
display(df_kpi)
# Confirm login event names — update KPI_EVENTS in params if they differ

✅ Post-install event names (fact_dsp_all): 30 rows


,os,event_name,conversions
0,ANDROID,visit_shop_1st,1391153
1,ANDROID,visit_shop,1389208
2,ANDROID,af_app_opened,948040
3,ANDROID,login,942400
4,IOS,visit_shop,283924
5,IOS,login,173360
6,ANDROID,cdn_download_start,128650
7,IOS,af_app_opened,127148
8,ANDROID,cdn_download_finished,125568
9,ANDROID,cdn_download_50p,125555


---
## Section 1: Campaign-Level Performance (Past 7 Days)

Metrics: CPI, Login CPA, D1/D7 ROAS — grouped by campaign and OS

In [98]:
#@title 1a. Campaign list — ID, name, goal, countries, status (Active = spend > 0 today)

q_campaigns = f"""
WITH all_campaigns AS (
  SELECT
    campaign_id,
    ANY_VALUE(campaign.title)                                            AS campaign_name,
    ANY_VALUE(campaign.os)                                               AS os,
    ANY_VALUE(campaign.goal)                                             AS goal,
    STRING_AGG(DISTINCT campaign.country ORDER BY campaign.country)      AS target_countries
  FROM `moloco-ae-view.athena.fact_dsp_core`
  WHERE product.app_market_bundle IN ({_bundle_fact_in})
    AND date_utc >= DATE_SUB(CURRENT_DATE(), INTERVAL 30 DAY)
  GROUP BY 1
),
today_active AS (
  SELECT campaign_id
  FROM `moloco-ae-view.athena.fact_dsp_core`
  WHERE product.app_market_bundle IN ({_bundle_fact_in})
    AND date_utc = CURRENT_DATE()
  GROUP BY 1
  HAVING SUM(gross_spend_usd) > 0
)
SELECT
  c.campaign_id,
  c.campaign_name,
  c.os,
  c.goal,
  c.target_countries,
  CASE WHEN t.campaign_id IS NOT NULL THEN 'Active' ELSE 'Inactive' END AS status
FROM all_campaigns c
LEFT JOIN today_active t USING (campaign_id)
ORDER BY status, os, campaign_id
"""

df_campaigns = run_query(q_campaigns, 'Campaign list')
display(df_campaigns)

✅ Campaign list: 29 rows


,campaign_id,campaign_name,os,goal,target_countries,status
0,nazpxG3J5MareHRz,[NEW]stonekey_launch_moloco_KR_And_tROAS_260303,ANDROID,OPTIMIZE_ROAS_FOR_APP_UA,KOR,Active
1,GdPe1hm9tPMaUhbt,[CKRC]stonekey_launch_moloco_KR_iOS_login_260303,IOS,OPTIMIZE_CPA_FOR_APP_UA,KOR,Active
2,EQCWerD5mEThZO4P,[NEW]stonekey_launch_moloco_KR_And_AEO(buy_pet_lv3)_260303,ANDROID,OPTIMIZE_CPA_FOR_APP_UA,KOR,Inactive
3,EcQ1HKvPWtQZkoEg,[NEW]stonekey_launch_moloco_ES_And_login_1st_260304,ANDROID,OPTIMIZE_CPA_FOR_APP_UA,ESP,Inactive
4,HvtBaZCEslMI1rJf,[NEW]stonekey_launch_moloco_JP_And_tROAS_260303,ANDROID,OPTIMIZE_ROAS_FOR_APP_UA,JPN,Inactive
5,I1UFqtzeIhRkfCrK,[NEW]stonekey_launch_moloco_TW_And_AEO(buy_pet_lv3)_260303,ANDROID,OPTIMIZE_CPA_FOR_APP_UA,TWN,Inactive
6,KVGI9Pc5DBldPeZw,[NEW]stonekey_softlaunch_moloco_HK_And_tROAS__260209,ANDROID,OPTIMIZE_ROAS_FOR_APP_UA,HKG,Inactive
7,MDGEIQDQ44rmAJGF,[NEW]stonekey_launch_moloco_JP_And_login_1st_260303,ANDROID,OPTIMIZE_CPA_FOR_APP_UA,JPN,Inactive
8,MsUKwNvjuzg3goIw,[NEW]stonekey_launch_moloco_US_And_login_1st_260303,ANDROID,OPTIMIZE_CPA_FOR_APP_UA,USA,Inactive
9,NMrtO3Y5EN9EAiYm,[NEW]stonekey_launch_moloco_DE_And_login_1st_260304,ANDROID,OPTIMIZE_CPA_FOR_APP_UA,DEU,Inactive


In [99]:
#@title 1a-2. Campaign duplication check — same OS + goal + country (Active, single-country only)

if not df_campaigns.empty:
    # Active + single-country targeting only
    df_check = df_campaigns[
        (df_campaigns['status'] == 'Active') &
        (~df_campaigns['target_countries'].str.contains(',', na=False))
    ].copy()

    if df_check.empty:
        print('⚠️ No active single-country campaigns found')
    else:
        df_check = df_check.rename(columns={'target_countries': 'country'})

        dup_groups = (df_check.groupby(['os', 'goal', 'country'])
                              .filter(lambda g: len(g) > 1))

        if dup_groups.empty:
            print('✅ No duplicate active single-country campaign pairs found (same OS + goal + country)')
        else:
            pairs = (dup_groups.groupby(['os', 'goal', 'country'])
                               .apply(lambda g: pd.Series({
                                   'campaign_ids':   ', '.join(g['campaign_id'].tolist()),
                                   'campaign_names': ' | '.join(g['campaign_name'].str[:30].tolist()),
                                   'count':          len(g)
                               }))
                               .reset_index()
                               .sort_values(['os', 'count'], ascending=[True, False]))

            print(f'⚠️ {len(pairs)} duplicate group(s) among active single-country campaigns:\n')
            display(pairs[['os', 'goal', 'country', 'count', 'campaign_ids', 'campaign_names']])
else:
    print('⚠️ Run 1a first')

✅ No duplicate active single-country campaign pairs found (same OS + goal + country)


In [100]:
#@title 1b. Campaign performance (L7D) — CPI, Login CPA, ROAS
# Spend/installs/ROAS from fact_dsp_core; Login CPA from fact_dsp_all (event.name + conversions)

q_perf = f"""
WITH spend AS (
  SELECT
    campaign_id,
    ANY_VALUE(campaign.title)    AS title,
    ANY_VALUE(campaign.os)       AS os,
    ANY_VALUE(campaign.country)  AS country,
    ANY_VALUE(campaign.goal)     AS goal,
    SUM(gross_spend_usd)         AS spend_L7D,
    SUM(installs)                AS installs_L7D,
    SUM(revenue_d1)              AS revenue_d1,
    SUM(revenue_d7)              AS revenue_d7
  FROM `moloco-ae-view.athena.fact_dsp_core`
  WHERE product.app_market_bundle IN ({_bundle_fact_in})
    AND date_utc BETWEEN DATE_SUB(CURRENT_DATE(), INTERVAL {ANALYSIS_WINDOW_DAYS} DAY)
                     AND DATE_SUB(CURRENT_DATE(), INTERVAL 1 DAY)
  GROUP BY 1
  HAVING SUM(gross_spend_usd) > 0
),
logins AS (
  SELECT
    campaign_id,
    SUM(conversions)             AS login_count
  FROM `moloco-ae-view.athena.fact_dsp_all`
  WHERE campaign_id IN (SELECT campaign_id FROM spend)
    AND DATE(timestamp_utc) BETWEEN DATE_SUB(CURRENT_DATE(), INTERVAL {ANALYSIS_WINDOW_DAYS} DAY)
                                AND DATE_SUB(CURRENT_DATE(), INTERVAL 1 DAY)
    AND LOWER(event.name) IN ({_kpi_in})
  GROUP BY 1
)
SELECT
  s.campaign_id,
  s.title,
  s.os,
  s.country,
  s.goal,
  s.spend_L7D,
  s.installs_L7D,
  COALESCE(l.login_count, 0)                             AS login_count,
  SAFE_DIVIDE(s.spend_L7D, NULLIF(s.installs_L7D, 0))   AS cpi,
  SAFE_DIVIDE(s.spend_L7D, NULLIF(l.login_count, 0))     AS login_cpa,
  SAFE_DIVIDE(s.revenue_d1, NULLIF(s.spend_L7D, 0))      AS roas_d1,
  SAFE_DIVIDE(s.revenue_d7, NULLIF(s.spend_L7D, 0))      AS roas_d7
FROM spend s
LEFT JOIN logins l USING (campaign_id)
ORDER BY s.spend_L7D DESC
"""

df_perf = run_query(q_perf, 'Campaign performance L7D')
display(df_perf.style.format({
    'spend_L7D':    '${:,.0f}',
    'installs_L7D': '{:,.0f}',
    'login_count':  '{:,.0f}',
    'cpi':          '${:.2f}',
    'login_cpa':    '${:.2f}',
    'roas_d1':      '{:.3f}',
    'roas_d7':      '{:.3f}',
}, na_rep='—'))

✅ Campaign performance L7D: 6 rows


,campaign_id,title,os,country,goal,spend_L7D,installs_L7D,login_count,cpi,login_cpa,roas_d1,roas_d7
0,nazpxG3J5MareHRz,[NEW]stonekey_launch_moloco_KR_And_tROAS_260303,ANDROID,KOR,OPTIMIZE_ROAS_FOR_APP_UA,"$22,446","1,685","1,790",$13.32,$12.54,0.348,0.784
1,GdPe1hm9tPMaUhbt,[CKRC]stonekey_launch_moloco_KR_iOS_login_260303,IOS,KOR,OPTIMIZE_CPA_FOR_APP_UA,"$7,797",569,588,$13.70,$13.26,0.231,0.394
2,huSii8ixFBbQjDXt,[NEW]stonekey_launch_moloco_WW3_And_tROAS_260304,ANDROID,LVA,OPTIMIZE_ROAS_FOR_APP_UA,"$4,605","1,261","1,286",$3.65,$3.58,0.051,0.088
3,unbUzFob1WCmTrVD,[NEW]stonekey_launch_moloco_WW1_And_tROAS_260303,ANDROID,BRA,OPTIMIZE_ROAS_FOR_APP_UA,"$1,012",389,399,$2.60,$2.54,0.276,0.347
4,cLMYkp0auemwutTK,[NEW]stonekey_launch_moloco_US_And_tROAS_260303,ANDROID,USA,OPTIMIZE_ROAS_FOR_APP_UA,$767,119,118,$6.44,$6.50,0.248,0.585
5,HvtBaZCEslMI1rJf,[NEW]stonekey_launch_moloco_JP_And_tROAS_260303,ANDROID,JPN,OPTIMIZE_ROAS_FOR_APP_UA,$29,10,15,$2.92,$1.95,0.061,0.061


In [101]:
#@title 1c. Chart — CPI & Login CPA by campaign, one chart per OS (ordered by CPI desc)

if not df_perf.empty:
    df_perf['geo'] = df_perf['country'].apply(lambda x: 'WW' if pd.isna(x) or ',' in str(x) else str(x))
    df_perf['base_label'] = df_perf['goal'].str.replace('OPTIMIZE_', '', regex=False) + ' | ' + df_perf['geo']

    # Append campaign_id in parentheses if base_label is duplicated within the same OS
    dup_mask = df_perf.duplicated(subset=['os', 'base_label'], keep=False)
    df_perf['x_label'] = df_perf.apply(
        lambda r: f"{r['base_label']} ({r['campaign_id']})" if dup_mask[r.name] else r['base_label'],
        axis=1
    )

    os_list = sorted(df_perf['os'].dropna().unique())
    fig = make_subplots(
        rows=len(os_list), cols=1,
        subplot_titles=[f'OS: {os}' for os in os_list],
        shared_xaxes=False,
        vertical_spacing=0.35 / max(len(os_list) - 1, 1)
    )

    for i, os_val in enumerate(os_list, start=1):
        sub = (df_perf[df_perf['os'] == os_val]
               .sort_values('cpi', ascending=False)
               .copy())

        fig.add_trace(go.Bar(
            name='CPI', x=sub['x_label'], y=sub['cpi'],
            marker_color='#636EFA', showlegend=(i == 1),
            hovertemplate='%{x}<br>CPI: $%{y:.2f}<extra>CPI</extra>'
        ), row=i, col=1)

        fig.add_trace(go.Bar(
            name='Login CPA', x=sub['x_label'], y=sub['login_cpa'],
            marker_color='#EF553B', showlegend=(i == 1),
            hovertemplate='%{x}<br>Login CPA: $%{y:.2f}<extra>Login CPA</extra>'
        ), row=i, col=1)

        fig.update_xaxes(tickangle=-40, row=i, col=1)
        fig.update_yaxes(title_text='USD', row=i, col=1)

    fig.update_layout(
        title=f'Stonekey — CPI & Login CPA by Campaign (L{ANALYSIS_WINDOW_DAYS}D)',
        barmode='group',
        height=500 * len(os_list),
        template='plotly_white',
        margin=dict(b=120)
    )
    fig.show()
else:
    print('⚠️ No data — run 1b query first')

In [102]:
#@title 1d. Chart — D1/D7 ROAS by campaign, one chart per OS (ordered by D7 ROAS desc)

if not df_perf.empty:
    fig2 = make_subplots(
        rows=len(os_list), cols=1,
        subplot_titles=[f'OS: {os}' for os in os_list],
        shared_xaxes=False,
        vertical_spacing=0.35 / max(len(os_list) - 1, 1)
    )

    for i, os_val in enumerate(os_list, start=1):
        sub = (df_perf[df_perf['os'] == os_val]
               .sort_values('roas_d7', ascending=False)
               .copy())

        fig2.add_trace(go.Bar(
            name='D1 ROAS', x=sub['x_label'], y=sub['roas_d1'],
            marker_color='#00CC96', showlegend=(i == 1),
            hovertemplate='%{x}<br>D1 ROAS: %{y:.3f}<extra>D1 ROAS</extra>'
        ), row=i, col=1)

        fig2.add_trace(go.Bar(
            name='D7 ROAS', x=sub['x_label'], y=sub['roas_d7'],
            marker_color='#AB63FA', showlegend=(i == 1),
            hovertemplate='%{x}<br>D7 ROAS: %{y:.3f}<extra>D7 ROAS</extra>'
        ), row=i, col=1)

        fig2.update_xaxes(tickangle=-40, row=i, col=1)
        fig2.update_yaxes(title_text='ROAS', row=i, col=1)

    fig2.update_layout(
        title=f'Stonekey — D1/D7 ROAS by Campaign (L{ANALYSIS_WINDOW_DAYS}D)',
        barmode='group',
        height=500 * len(os_list),
        template='plotly_white',
        margin=dict(b=120)
    )
    fig2.show()
else:
    print('⚠️ No data — run 1b query first')

### 1e. Daily Timeline — Mar 5 to Mar 11

In [103]:
#@title 1e. Daily timeline query — Mar 5–11, active campaigns only

TIMELINE_START = '2026-03-05'
TIMELINE_END   = '2026-03-11'

# Get active campaign IDs from 1a result
if 'df_campaigns' not in dir() or df_campaigns.empty:
    print('⚠️ Run 1a first to populate df_campaigns')
else:
    active_campaign_ids = df_campaigns[df_campaigns['status'] == 'Active']['campaign_id'].tolist()
    if not active_campaign_ids:
        print('⚠️ No active campaigns found in df_campaigns')
    else:
        _active_ids_in = ', '.join(f"'{c}'" for c in active_campaign_ids)

        q_timeline = f"""
        WITH daily_spend AS (
          SELECT
            date_utc,
            campaign_id,
            ANY_VALUE(campaign.title)                                    AS campaign_name,
            ANY_VALUE(campaign.os)                                       AS os,
            ANY_VALUE(campaign.country)                                  AS country,
            ANY_VALUE(campaign.goal)                                     AS goal,
            SUM(gross_spend_usd)                                         AS spend,
            SUM(installs)                                                AS installs,
            SUM(revenue_d1)                                              AS revenue_d1,
            SUM(revenue_d7)                                              AS revenue_d7
          FROM `moloco-ae-view.athena.fact_dsp_core`
          WHERE campaign_id IN ({_active_ids_in})
            AND date_utc BETWEEN '{TIMELINE_START}' AND '{TIMELINE_END}'
          GROUP BY 1, 2
        ),
        daily_logins AS (
          SELECT
            DATE(timestamp_utc)                                          AS date_utc,
            campaign_id,
            SUM(conversions)                                             AS login_count
          FROM `moloco-ae-view.athena.fact_dsp_all`
          WHERE campaign_id IN ({_active_ids_in})
            AND DATE(timestamp_utc) BETWEEN '{TIMELINE_START}' AND '{TIMELINE_END}'
            AND LOWER(event.name) IN ({_kpi_in})
          GROUP BY 1, 2
        )
        SELECT
          s.date_utc,
          s.campaign_id,
          s.campaign_name,
          s.os,
          s.country,
          s.goal,
          s.spend,
          s.installs,
          COALESCE(l.login_count, 0)                                     AS login_count,
          SAFE_DIVIDE(s.spend, NULLIF(s.installs, 0))                    AS cpi,
          SAFE_DIVIDE(s.spend, NULLIF(l.login_count, 0))                 AS login_cpa,
          SAFE_DIVIDE(s.revenue_d1, NULLIF(s.spend, 0))                  AS roas_d1,
          SAFE_DIVIDE(s.revenue_d7, NULLIF(s.spend, 0))                  AS roas_d7
        FROM daily_spend s
        LEFT JOIN daily_logins l USING (date_utc, campaign_id)
        ORDER BY s.os, s.campaign_id, s.date_utc
        """

        df_timeline = run_query(q_timeline, f'Daily timeline {TIMELINE_START} → {TIMELINE_END} ({len(active_campaign_ids)} active campaigns)')
        display(df_timeline.style.format({
            'spend':       '${:,.0f}',
            'installs':    '{:,.0f}',
            'login_count': '{:,.0f}',
            'cpi':         '${:.2f}',
            'login_cpa':   '${:.2f}',
            'roas_d1':     '{:.3f}',
            'roas_d7':     '{:.3f}',
        }, na_rep='—'))

✅ Daily timeline 2026-03-05 → 2026-03-11 (2 active campaigns): 14 rows


,date_utc,campaign_id,campaign_name,os,country,goal,spend,installs,login_count,cpi,login_cpa,roas_d1,roas_d7
0,2026-03-05,nazpxG3J5MareHRz,[NEW]stonekey_launch_moloco_KR_And_tROAS_260303,ANDROID,KOR,OPTIMIZE_ROAS_FOR_APP_UA,"$30,360",567,569,$53.55,$53.36,0.105,0.329
1,2026-03-06,nazpxG3J5MareHRz,[NEW]stonekey_launch_moloco_KR_And_tROAS_260303,ANDROID,KOR,OPTIMIZE_ROAS_FOR_APP_UA,"$22,508",565,523,$39.84,$43.04,0.144,0.459
2,2026-03-07,nazpxG3J5MareHRz,[NEW]stonekey_launch_moloco_KR_And_tROAS_260303,ANDROID,KOR,OPTIMIZE_ROAS_FOR_APP_UA,"$18,388",687,628,$26.77,$29.28,0.254,0.853
3,2026-03-08,nazpxG3J5MareHRz,[NEW]stonekey_launch_moloco_KR_And_tROAS_260303,ANDROID,KOR,OPTIMIZE_ROAS_FOR_APP_UA,"$30,040",775,742,$38.76,$40.49,0.137,0.467
4,2026-03-09,nazpxG3J5MareHRz,[NEW]stonekey_launch_moloco_KR_And_tROAS_260303,ANDROID,KOR,OPTIMIZE_ROAS_FOR_APP_UA,"$13,594",502,492,$27.08,$27.63,0.206,0.775
5,2026-03-10,nazpxG3J5MareHRz,[NEW]stonekey_launch_moloco_KR_And_tROAS_260303,ANDROID,KOR,OPTIMIZE_ROAS_FOR_APP_UA,"$11,267",512,511,$22.01,$22.05,0.172,0.718
6,2026-03-11,nazpxG3J5MareHRz,[NEW]stonekey_launch_moloco_KR_And_tROAS_260303,ANDROID,KOR,OPTIMIZE_ROAS_FOR_APP_UA,"$12,182",593,566,$20.54,$21.52,0.182,0.809
7,2026-03-05,GdPe1hm9tPMaUhbt,[CKRC]stonekey_launch_moloco_KR_iOS_login_260303,IOS,KOR,OPTIMIZE_CPA_FOR_APP_UA,"$6,089",453,456,$13.44,$13.35,0.304,0.738
8,2026-03-06,GdPe1hm9tPMaUhbt,[CKRC]stonekey_launch_moloco_KR_iOS_login_260303,IOS,KOR,OPTIMIZE_CPA_FOR_APP_UA,"$4,966",320,325,$15.52,$15.28,0.138,0.349
9,2026-03-07,GdPe1hm9tPMaUhbt,[CKRC]stonekey_launch_moloco_KR_iOS_login_260303,IOS,KOR,OPTIMIZE_CPA_FOR_APP_UA,"$6,277",456,464,$13.77,$13.53,0.167,0.609


In [104]:
#@title 1f. Chart — Daily timeline by campaign (CPI, Login CPA, Spend, Installs)

if 'df_timeline' not in dir() or df_timeline.empty:
    print('⚠️ Run 1e first')
else:
    # Build one stable label per campaign from df_campaigns (not df_timeline)
    df_cam_meta = df_campaigns[df_campaigns['status'] == 'Active'].copy()
    df_cam_meta['geo'] = df_cam_meta['target_countries'].apply(
        lambda x: 'WW' if pd.isna(x) or ',' in str(x) else str(x)
    )
    df_cam_meta['base_label'] = (df_cam_meta['goal'].str.replace('OPTIMIZE_', '', regex=False)
                                 + ' | ' + df_cam_meta['geo']
                                 + ' (' + df_cam_meta['os'] + ')')
    base_dup = df_cam_meta['base_label'].duplicated(keep=False)
    df_cam_meta['cam_label'] = df_cam_meta.apply(
        lambda r: f"{r['base_label']} [{r['campaign_id']}]" if base_dup[r.name] else r['base_label'],
        axis=1
    )
    label_map = df_cam_meta.set_index('campaign_id')['cam_label'].to_dict()
    df_timeline['cam_label'] = df_timeline['campaign_id'].map(label_map).fillna(df_timeline['campaign_id'])

    metrics = [
        ('cpi',       'CPI ($)'),
        ('login_cpa', 'Login CPA ($)'),
        ('spend',     'Spend ($)'),
        ('installs',  'Installs'),
    ]

    campaigns = df_cam_meta[['campaign_id', 'cam_label']].drop_duplicates().sort_values('cam_label')

    fig_tl = make_subplots(
        rows=len(metrics), cols=1,
        subplot_titles=[m[1] for m in metrics],
        shared_xaxes=True,
        vertical_spacing=0.07
    )

    colors = px.colors.qualitative.Plotly
    for c_i, (_, row) in enumerate(campaigns.iterrows()):
        cid, clabel = row['campaign_id'], row['cam_label']
        sub = df_timeline[df_timeline['campaign_id'] == cid].sort_values('date_utc')
        color = colors[c_i % len(colors)]

        for r_i, (col, label) in enumerate(metrics, start=1):
            fig_tl.add_trace(go.Scatter(
                x=sub['date_utc'], y=sub[col],
                name=clabel, mode='lines+markers',
                line=dict(color=color),
                showlegend=(r_i == 1),
                legendgroup=cid,
                hovertemplate=f'<b>{clabel}</b><br>%{{x}}<br>{label}: %{{y:,.2f}}<extra></extra>'
            ), row=r_i, col=1)

    for r_i, (_, label) in enumerate(metrics, start=1):
        fig_tl.update_yaxes(title_text=label, row=r_i, col=1)

    fig_tl.update_layout(
        title=f'Stonekey — Daily Trend by Campaign ({TIMELINE_START} → {TIMELINE_END})',
        height=260 * len(metrics),
        template='plotly_white',
        hovermode='closest',
        legend=dict(orientation='v', x=1.02, y=1)
    )
    fig_tl.update_xaxes(dtick='D1', tickformat='%m/%d', row=len(metrics), col=1)
    fig_tl.show()

---
## Section 2: Purchasing Behavior — Revenue Timeline

Both 2a and 2b use `prod_stream_view.cv` as the single source of truth for revenue, ensuring consistent percentages across the snapshot table and the daily curve.

### 2a. Cohort Snapshot (D1/D3/D7/D14/D30) from `cv`

In [105]:
#@title 2a. Cohort snapshot — D1/D3/D7 by OS × geo (GLOBAL + KOR/USA/JPN)

q_cohort = f"""
WITH base AS (
  SELECT
    cv.pb.device.os                                                  AS os,
    cv.pb.device.country                                             AS country,
    TIMESTAMP_DIFF(timestamp, cv.pb.event.install_at, DAY)           AS dsi,
    cv.revenue_usd.amount                                            AS rev
  FROM `focal-elf-631.prod_stream_view.cv`
  WHERE cv.pb.app.bundle IN ({_bundle_cv_in})
    AND DATE(timestamp) BETWEEN DATE_SUB(CURRENT_DATE(), INTERVAL {COHORT_WINDOW_DAYS} DAY)
                            AND CURRENT_DATE()
    AND cv.pb.moloco.attributed = TRUE
    AND cv.revenue_usd.amount > 0
    AND DATE(cv.pb.event.install_at) >= DATE_SUB(CURRENT_DATE(), INTERVAL {COHORT_WINDOW_DAYS} DAY)
    AND TIMESTAMP_DIFF(timestamp, cv.pb.event.install_at, DAY) BETWEEN 0 AND {COHORT_MAX_DAY}
),
global_agg AS (
  SELECT
    'GLOBAL'                                          AS geo,
    os,
    SUM(CASE WHEN dsi <= 1 THEN rev ELSE 0 END)       AS rev_d1,
    SUM(CASE WHEN dsi <= 3 THEN rev ELSE 0 END)       AS rev_d3,
    SUM(rev)                                          AS rev_d7
  FROM base
  GROUP BY os
),
country_agg AS (
  SELECT
    country                                           AS geo,
    os,
    SUM(CASE WHEN dsi <= 1 THEN rev ELSE 0 END)       AS rev_d1,
    SUM(CASE WHEN dsi <= 3 THEN rev ELSE 0 END)       AS rev_d3,
    SUM(rev)                                          AS rev_d7
  FROM base
  WHERE country IN ({_cohort_ctry_in})
  GROUP BY country, os
)
SELECT * FROM global_agg
UNION ALL
SELECT * FROM country_agg
ORDER BY geo, os
"""

df_cohort = run_query(q_cohort, 'Cohort snapshot D0–D7 (global + country)')

if not df_cohort.empty:
    df_cohort['pct_d1'] = df_cohort['rev_d1'] / df_cohort['rev_d7'].replace(0, np.nan)
    df_cohort['pct_d3'] = df_cohort['rev_d3'] / df_cohort['rev_d7'].replace(0, np.nan)

    print('⚠️  Note: recent cohorts are partially baked — percentages will shift as more revenue arrives')
    display(df_cohort[['geo', 'os', 'rev_d1', 'rev_d3', 'rev_d7', 'pct_d1', 'pct_d3']]
            .style.format({
                'rev_d1': '${:,.0f}', 'rev_d3': '${:,.0f}', 'rev_d7': '${:,.0f}',
                'pct_d1': '{:.1%}',   'pct_d3': '{:.1%}',
            }))

✅ Cohort snapshot D0–D7 (global + country): 8 rows
⚠️  Note: recent cohorts are partially baked — percentages will shift as more revenue arrives


,geo,os,rev_d1,rev_d3,rev_d7,pct_d1,pct_d3
0,GLOBAL,ANDROID,"$65,979","$95,010","$143,910",45.8%,66.0%
1,GLOBAL,IOS,"$13,671","$18,255","$24,914",54.9%,73.3%
2,JPN,ANDROID,$737,$949,"$1,148",64.2%,82.7%
3,JPN,IOS,$709,$967,"$1,268",55.9%,76.2%
4,KOR,ANDROID,"$54,458","$79,381","$123,652",44.0%,64.2%
5,KOR,IOS,"$10,568","$14,082","$19,319",54.7%,72.9%
6,USA,ANDROID,"$1,661","$2,180","$2,663",62.4%,81.9%
7,USA,IOS,$441,$474,$490,90.1%,96.7%


### 2b. Continuous Daily Revenue Curve from `cv` (D0–D30)

In [106]:
#@title 2b. Daily revenue curve from cv (D0–D7, global + KOR/USA/JPN)

q_curve = f"""
WITH base AS (
  SELECT
    cv.pb.device.os                                                  AS os,
    cv.pb.device.country                                             AS country,
    TIMESTAMP_DIFF(timestamp, cv.pb.event.install_at, DAY)           AS days_since_install,
    cv.revenue_usd.amount                                            AS rev
  FROM `focal-elf-631.prod_stream_view.cv`
  WHERE cv.pb.app.bundle IN ({_bundle_cv_in})
    AND DATE(timestamp) BETWEEN DATE_SUB(CURRENT_DATE(), INTERVAL {COHORT_WINDOW_DAYS} DAY)
                            AND CURRENT_DATE()
    AND cv.pb.moloco.attributed = TRUE
    AND cv.revenue_usd.amount > 0
    AND DATE(cv.pb.event.install_at) >= DATE_SUB(CURRENT_DATE(), INTERVAL {COHORT_WINDOW_DAYS} DAY)
    AND TIMESTAMP_DIFF(timestamp, cv.pb.event.install_at, DAY) BETWEEN 0 AND {COHORT_MAX_DAY}
)
SELECT 'GLOBAL' AS geo, os, days_since_install, SUM(rev) AS revenue_usd
FROM base
GROUP BY os, days_since_install

UNION ALL

SELECT country AS geo, os, days_since_install, SUM(rev) AS revenue_usd
FROM base
WHERE country IN ({_cohort_ctry_in})
GROUP BY country, os, days_since_install

ORDER BY geo, os, days_since_install
"""

df_curve_raw = run_query(q_curve, 'Daily revenue curve D0–D7 (global + country)')

if not df_curve_raw.empty:
    all_days = pd.DataFrame({'days_since_install': range(COHORT_MAX_DAY + 1)})
    plot_data = []

    for (geo, os_val), grp in df_curve_raw.groupby(['geo', 'os']):
        sub = (grp[['days_since_install', 'revenue_usd']]
               .merge(all_days, on='days_since_install', how='right')
               .fillna(0)
               .sort_values('days_since_install'))
        sub['cumrev'] = sub['revenue_usd'].cumsum()

        # Normalise by D7 total from 2a for the same geo+os
        d7 = df_cohort.loc[(df_cohort['geo'] == geo) & (df_cohort['os'] == os_val), 'rev_d7'].values
        total_rev = d7[0] if len(d7) > 0 else sub['cumrev'].max()

        sub['cumrev_pct'] = sub['cumrev'] / total_rev if total_rev > 0 else 0
        sub['geo'] = geo
        sub['os']  = os_val
        sub['line'] = f'{geo} | {os_val}'
        plot_data.append(sub)

    df_curve = pd.concat(plot_data, ignore_index=True)
    print(f'✅ Daily curve built: {df_curve["line"].nunique()} series')

✅ Daily revenue curve D0–D7 (global + country): 64 rows
✅ Daily curve built: 8 series


In [107]:
#@title 2b. Chart — Cumulative revenue curve D0–D7, one subplot per geo (Android vs iOS)

if 'df_curve' not in dir() or df_curve.empty:
    print('⚠️ Run 2b query first')
else:
    geos = ['GLOBAL'] + COHORT_COUNTRIES  # GLOBAL first, then countries
    geos = [g for g in geos if g in df_curve['geo'].unique()]

    fig3 = make_subplots(
        rows=len(geos), cols=1,
        subplot_titles=geos,
        shared_xaxes=True,
        vertical_spacing=0.08
    )

    os_colors = {'ANDROID': '#636EFA', 'IOS': '#EF553B'}

    for row_i, geo in enumerate(geos, start=1):
        sub_geo = df_curve[df_curve['geo'] == geo]

        for os_val in sorted(sub_geo['os'].unique()):
            sub = sub_geo[sub_geo['os'] == os_val].sort_values('days_since_install')
            color = os_colors.get(os_val, '#00CC96')

            fig3.add_trace(go.Scatter(
                x=sub['days_since_install'], y=sub['cumrev_pct'],
                name=os_val, mode='lines+markers',
                line=dict(color=color),
                showlegend=(row_i == 1),
                legendgroup=os_val,
                hovertemplate=f'Day %{{x}}<br>Cumulative: %{{y:.1%}}<extra>{geo} | {os_val}</extra>'
            ), row=row_i, col=1)

        # Reference lines (only add annotations on first row to avoid clutter)
        for day in [1, 3, 7]:
            fig3.add_vline(x=day, line_dash='dash', line_color='gray', opacity=0.4, row=row_i, col=1)
        fig3.add_hline(y=0.50, line_dash='dot', line_color='orange', opacity=0.6, row=row_i, col=1)
        fig3.update_yaxes(title_text='% of D7 Rev', tickformat='.0%', row=row_i, col=1)

    fig3.update_xaxes(title_text='Days Since Install', dtick=1, range=[0, COHORT_MAX_DAY], row=len(geos), col=1)
    fig3.update_layout(
        title=f'Stonekey — Cumulative Revenue D0–D{COHORT_MAX_DAY} by Geo (partially baked)',
        height=280 * len(geos),
        template='plotly_white',
        hovermode='closest',
        legend=dict(orientation='v', x=1.02, y=1)
    )
    fig3.show()

### 2c. Cumulative Payer Curve from `cv` (D0–D30)

Counts **distinct payers** (unique device IDs with ≥1 revenue event) by their first purchase day since install.
X-axis = days since install, Y-axis = cumulative % of total D7 payers.

In [111]:
#@title 2c. Daily payer curve from cv — distinct payers by first purchase day (global + KOR/USA/JPN)

q_payers = f"""
WITH base AS (
  SELECT
    cv.pb.device.os                                                         AS os,
    cv.pb.device.country                                                    AS country,
    cv.pb.device.ifa                                                        AS device_id,
    MIN(TIMESTAMP_DIFF(timestamp, cv.pb.event.install_at, DAY))             AS first_purchase_day
  FROM `focal-elf-631.prod_stream_view.cv`
  WHERE cv.pb.app.bundle IN ({_bundle_cv_in})
    AND DATE(timestamp) BETWEEN DATE_SUB(CURRENT_DATE(), INTERVAL {COHORT_WINDOW_DAYS} DAY)
                            AND CURRENT_DATE()
    AND cv.pb.moloco.attributed = TRUE
    AND cv.revenue_usd.amount > 0
    AND DATE(cv.pb.event.install_at) >= DATE_SUB(CURRENT_DATE(), INTERVAL {COHORT_WINDOW_DAYS} DAY)
    AND TIMESTAMP_DIFF(timestamp, cv.pb.event.install_at, DAY) BETWEEN 0 AND {COHORT_MAX_DAY}
  GROUP BY os, country, device_id
)
SELECT 'GLOBAL' AS geo, os, first_purchase_day AS days_since_install, COUNT(*) AS new_payers
FROM base
GROUP BY os, first_purchase_day

UNION ALL

SELECT country AS geo, os, first_purchase_day AS days_since_install, COUNT(*) AS new_payers
FROM base
WHERE country IN ({_cohort_ctry_in})
GROUP BY country, os, first_purchase_day

ORDER BY geo, os, days_since_install
"""

df_payers_raw = run_query(q_payers, 'Daily payer curve D0–D7 (global + country)')

if not df_payers_raw.empty:
    all_days = pd.DataFrame({'days_since_install': range(COHORT_MAX_DAY + 1)})
    payer_plot_data = []

    for (geo, os_val), grp in df_payers_raw.groupby(['geo', 'os']):
        sub = (grp[['days_since_install', 'new_payers']]
               .merge(all_days, on='days_since_install', how='right')
               .fillna(0)
               .sort_values('days_since_install'))
        sub['cum_payers'] = sub['new_payers'].cumsum()

        total_payers = sub['cum_payers'].max()
        sub['cum_pct'] = sub['cum_payers'] / total_payers if total_payers > 0 else 0
        sub['geo']  = geo
        sub['os']   = os_val
        sub['line'] = f'{geo} | {os_val}'
        payer_plot_data.append(sub)

    df_payers = pd.concat(payer_plot_data, ignore_index=True)
    print(f'✅ Payer curve built: {df_payers["line"].nunique()} series')

    # Print total payer counts per geo × OS
    summary = (df_payers.groupby(['geo', 'os'])['cum_payers'].max()
                        .rename('total_payers_D7').reset_index())
    display(summary)

✅ Daily payer curve D0–D7 (global + country): 61 rows
✅ Payer curve built: 8 series


,geo,os,total_payers_D7
0,GLOBAL,ANDROID,5009
1,GLOBAL,IOS,932
2,JPN,ANDROID,134
3,JPN,IOS,101
4,KOR,ANDROID,2935
5,KOR,IOS,654
6,USA,ANDROID,206
7,USA,IOS,57


In [112]:
#@title 2c. Chart — Cumulative payer curve D0–D7, one subplot per geo (Android vs iOS)

if 'df_payers' not in dir() or df_payers.empty:
    print('⚠️ Run 2c query first')
else:
    geos_p = ['GLOBAL'] + COHORT_COUNTRIES
    geos_p = [g for g in geos_p if g in df_payers['geo'].unique()]

    fig_p = make_subplots(
        rows=len(geos_p), cols=1,
        subplot_titles=geos_p,
        shared_xaxes=True,
        vertical_spacing=0.08
    )

    os_colors = {'ANDROID': '#636EFA', 'IOS': '#EF553B'}

    for row_i, geo in enumerate(geos_p, start=1):
        sub_geo = df_payers[df_payers['geo'] == geo]

        for os_val in sorted(sub_geo['os'].unique()):
            sub = sub_geo[sub_geo['os'] == os_val].sort_values('days_since_install')
            color = os_colors.get(os_val, '#00CC96')
            total = int(sub['cum_payers'].max())

            fig_p.add_trace(go.Scatter(
                x=sub['days_since_install'], y=sub['cum_pct'],
                name=os_val, mode='lines+markers',
                line=dict(color=color),
                showlegend=(row_i == 1),
                legendgroup=os_val,
                hovertemplate=(
                    f'Day %{{x}}<br>Cumulative: %{{y:.1%}}<br>'
                    f'(of {total:,} total D{COHORT_MAX_DAY} payers)'
                    f'<extra>{geo} | {os_val}</extra>'
                )
            ), row=row_i, col=1)

        for day in [1, 3, 7]:
            fig_p.add_vline(x=day, line_dash='dash', line_color='gray', opacity=0.4, row=row_i, col=1)
        fig_p.add_hline(y=0.50, line_dash='dot', line_color='orange', opacity=0.6, row=row_i, col=1)
        fig_p.update_yaxes(title_text='% of D7 Payers', tickformat='.0%', row=row_i, col=1)

    fig_p.update_xaxes(title_text='Days Since Install', dtick=1, range=[0, COHORT_MAX_DAY], row=len(geos_p), col=1)
    fig_p.update_layout(
        title=f'Stonekey — Cumulative Payer Curve D0–D{COHORT_MAX_DAY} by Geo (partially baked)',
        height=280 * len(geos_p),
        template='plotly_white',
        hovermode='closest',
        legend=dict(orientation='v', x=1.02, y=1)
    )
    fig_p.show()

---
## Section 3: D1/D3 Model Recommendation

**Eligibility Logic:**
- **D1 model**: ≥50% of D30 revenue arrives within D1
- **D3 model**: ≥50% of D30 revenue arrives within D3 (but <50% within D1)
- **D7 default**: neither threshold met

Source: `fact_dsp_core` cohort snapshot (Section 2a)

In [113]:
#@title 3. D1/D3 Model Eligibility Summary (by geo × OS)

def get_recommendation(pct_d1, pct_d3):
    if pd.isna(pct_d1) or pd.isna(pct_d3):
        return '⚠️ Insufficient data'
    if pct_d1 >= D1_THRESHOLD:
        return '✅ D1 model eligible'
    elif pct_d3 >= D3_THRESHOLD:
        return '🔶 D3 model eligible'
    else:
        return '⬜ D7 model (default) — insufficient early purchases'

if not df_cohort.empty and 'pct_d1' in df_cohort.columns:
    df_rec = df_cohort[['geo', 'os', 'pct_d1', 'pct_d3']].copy()
    df_rec['recommendation'] = df_rec.apply(
        lambda r: get_recommendation(r['pct_d1'], r['pct_d3']), axis=1
    )

    print(f'\n=== D1/D3 Model Eligibility (D0–D{COHORT_MAX_DAY} cv data, partially baked) ===')
    display(df_rec.style.format({'pct_d1': '{:.1%}', 'pct_d3': '{:.1%}'}))

    print(f'\n⚠️  Caveat: cohort data is not fully baked — re-run once D7+ cohorts mature.')
    print(f'\n📋 For formal eligibility requests, submit via the tracker:')
    print(f'   {ELIGIBILITY_TRACKER_URL}')
else:
    print('⚠️ Run Section 2a first to populate cohort data')


=== D1/D3 Model Eligibility (D0–D7 cv data, partially baked) ===


,geo,os,pct_d1,pct_d3,recommendation
0,GLOBAL,ANDROID,45.8%,66.0%,🔶 D3 model eligible
1,GLOBAL,IOS,54.9%,73.3%,✅ D1 model eligible
2,JPN,ANDROID,64.2%,82.7%,✅ D1 model eligible
3,JPN,IOS,55.9%,76.2%,✅ D1 model eligible
4,KOR,ANDROID,44.0%,64.2%,🔶 D3 model eligible
5,KOR,IOS,54.7%,72.9%,✅ D1 model eligible
6,USA,ANDROID,62.4%,81.9%,✅ D1 model eligible
7,USA,IOS,90.1%,96.7%,✅ D1 model eligible



⚠️  Caveat: cohort data is not fully baked — re-run once D7+ cohorts mature.

📋 For formal eligibility requests, submit via the tracker:
   https://docs.google.com/spreadsheets/d/1w8StJ19HpuPZ8kj4oA3SD_To5sEoEVpoZce8gpkthQY/edit?gid=1375953392
